<a href="https://colab.research.google.com/github/goAustin/ML2025/blob/main/Homework5_Finetuning_is_Powerful.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Check GPU Availability

In [1]:
!nvidia-smi

Sun Aug 16 05:55:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Installation

In [1]:
%%capture
# Compatible package set for the pinned Unsloth/Transformers notebook
!pip install --no-deps bitsandbytes xformers==0.0.29 peft==0.14.0 trl==0.14.0 triton
!pip install --no-deps cut_cross_entropy
!pip install sentencepiece datasets huggingface_hub hf_transfer
!pip install unsloth==2025.2.15 unsloth_zoo==2025.2.7
!pip install --no-deps --force-reinstall accelerate==1.4.0
!pip install transformers==4.49.0
!pip install -U --no-cache-dir "protobuf>=5.26.1,<6"

In [2]:
import os
import subprocess

repo_dir = "/content/ML_Spring2025_HW5"
if not os.path.isdir(os.path.join(repo_dir, ".git")):
    subprocess.run(
        ["git", "clone", "https://github.com/ericsunkuan/ML_Spring2025_HW5.git", repo_dir],
        check=True,
    )
else:
    print("Dataset repository already exists; using it.")

Dataset repository already exists; using it.


### Unsloth

#### Note : Changing the model is against the Rules of Homework 5 !!!

### Initialize the LLM

In [3]:
# Compatibility shim: Unsloth 2025.2.15 imports this symbol from the old location.
from accelerate.utils.dataclasses import FP8BackendType
import accelerate.utils as accelerate_utils
accelerate_utils.FP8BackendType = FP8BackendType

from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-2-7b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: Tesla T4. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
################# TODO : Tweak the LoRA adapter hyperparameters here.  #####################

model = FastLanguageModel.get_peft_model(
    model,
    r = 8, ### TODO : Choose any number > 0 ! Common values are 4, 8, 16, 32, 64, 128. Higher ranks allow more expressive power but also increase parameter count.
    lora_alpha = 16,  ### TODO : Choose any number > 0 ! Suggested 4, 8, 16, 32, 64, 128


################# TODO  ####################################################################
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.2.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",  ### Use llama-3.1 template for better performance here
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass


# Dataset Preperation (Loading and Refining)

## Data Filtering & Sorting

In [6]:
from datasets import load_dataset, Dataset, load_from_disk

# Load the dataset from Hugging Face
dataset = load_from_disk("/content/ML_Spring2025_HW5/fastchat_alpaca_52k")

# ---------------------------
# Add a "text" field to each example
# ---------------------------
# This function extracts the first assistant message from the conversation
def add_text_field(example):
    # Extract the first message where role == 'assistant'
    assistant_texts = [msg["content"] for msg in example["conversations"] if msg["role"] == "assistant"]
    text = assistant_texts[0] if assistant_texts else ""
    return {"text": text}

# Map the function over the dataset to add the "text" column.
dataset = dataset.map(add_text_field)

# Print the dataset structure to confirm the new feature.
print(dataset)


# ---------------------------
#################### TODO : Define a helper function for computing conversation length ###############

# The default "conversation length" here refers to the length of the input (human) and output (gpt), you can modify it at your will

def compute_conversation_length(example):
    # Compute total word count across all messages in the 'conversations' field
    return sum(len(message["content"].split()) for message in example["conversations"])


#################### TODO ############################################################################

# ---------------------------
# Simple Sorting Method  (Default)
# ---------------------------
# Sort the dataset from shortest to longest conversation (by word count)
sorted_dataset_simple_list = sorted(dataset, key=compute_conversation_length, reverse=False)

# Convert back to a Dataset object
sorted_dataset_simple = Dataset.from_list(sorted_dataset_simple_list)

print("\nTop examples sorted by simple conversation length:")
for entry in sorted_dataset_simple.select(range(5)):
    print(f"ID: {entry['id']}, Conversation Length: {compute_conversation_length(entry)}")
# ---------------------------



############## Advanced Sorting Method (TODO : Modify the sorting key ##################
# ---------------------------
# Default : Sorting based on Combining conversation length with the 'score' field using a weighted sum.
# Here, we multiply the score by 10 and add it to the conversation length.
def advanced_sort_key(example):
    conversation_len = compute_conversation_length(example)
    score = example["score"]
    return 1e-5 * conversation_len + score * 1

####################################### TODO ###########################################

sorted_dataset_advanced_list = sorted(dataset, key=advanced_sort_key, reverse=True)
# Convert back to a Dataset object
sorted_dataset_advanced = Dataset.from_list(sorted_dataset_advanced_list)

print("\nTop examples sorted by advanced key (combination of conversation length and score):")
for entry in sorted_dataset_advanced.select(range(5)):
    print(f"ID: {entry['id']}, Advanced Key Value: {advanced_sort_key(entry)}")


Dataset({
    features: ['id', 'conversations', 'score', 'text'],
    num_rows: 52002
})

Top examples sorted by simple conversation length:
ID: identity_23488, Conversation Length: 22
ID: identity_717, Conversation Length: 23
ID: identity_1347, Conversation Length: 23
ID: identity_1790, Conversation Length: 23
ID: identity_2502, Conversation Length: 23

Top examples sorted by advanced key (combination of conversation length and score):
ID: identity_23370, Advanced Key Value: 5.00049
ID: identity_37225, Advanced Key Value: 5.00047
ID: identity_40362, Advanced Key Value: 5.0004
ID: identity_8015, Advanced Key Value: 5.00037
ID: identity_30813, Advanced Key Value: 5.00037


#### Note : You are limited to use 100 sorted data among the 1000 data in the given dataset, no more than 100 data is allowed for training !!!

In [7]:
################# TODO : select the simple or advanced dataset for training ##############

dataset_used = "sorted_dataset_simple" #sorted_dataset_advanced

################# TODO ###################################################################

if dataset_used == "sorted_dataset_simple":
    train_dataset = sorted_dataset_simple.select(range(0,100))    ### You can also select from the middle, e.g. sorted_dataset_simple.select(range(50,150))
else:
    train_dataset = sorted_dataset_advanced.select(range(0,100))

from unsloth.chat_templates import standardize_sharegpt
train_dataset = standardize_sharegpt(train_dataset)
train_dataset = train_dataset.map(formatting_prompts_func, batched = True,)

Standardizing format:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

# Dataset Visualize

In [8]:
dataset[5]["conversations"]

[{'content': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. ### Instruction: Identify the odd one out. ### Input: Twitter, Instagram, Telegram',
  'role': 'user'},
 {'content': 'Telegram', 'role': 'assistant'}]

And we see how the chat template transformed these conversations.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

In [9]:
dataset[5]["text"]

'Telegram'

## Training

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported


################# TODO : Tweak the training hyperparameters here.  #####################


training_config = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_steps": 10,
    "num_train_epochs": 2,
    "learning_rate": 2e-4,
    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "seed": 3407,   ### Do not modify the seed for reproducibility
}


################# TODO #################################################################

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = training_config["per_device_train_batch_size"],
        gradient_accumulation_steps = training_config["gradient_accumulation_steps"],
        warmup_steps = training_config["warmup_steps"],
        num_train_epochs = training_config["num_train_epochs"], # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = training_config["learning_rate"],
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = training_config["optim"],
        weight_decay = training_config["weight_decay"],
        lr_scheduler_type = training_config["lr_scheduler_type"],
        seed = training_config["seed"],
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Map (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [11]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 100 | Num Epochs = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 24
 "-____-"     Number of trainable parameters = 19,988,480


Step,Training Loss
1,1.762600
2,1.996500
3,1.384800
4,1.196400
5,1.562500
6,1.006600
7,1.083600
8,0.869300
9,0.792900
10,1.075200


#### TODO : Curriculum Training  (Optional)
start training the LLM with “easier” examples (e.g., shorter, clearer conversations) and progressively introduce more complex ones.

The total data amount used to train should still not exceed 100 data.

In [15]:
############## TODO : Curriculum Training  ######################

### E.g.
### Step 1. Train on sorted_dataset_simple
### Step 2. Train on sorted_dataset_advanced

<a name="Inference"></a>
## Inference


In [17]:
def parse_true_output(text):
    """
    Extracts the true assistant output from the decoded model output.

    It looks for the assistant header token:
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    and extracts everything after it until the first occurrence of "<|eot_id|>".
    If the assistant header is not found, it falls back to the last occurrence
    of "<|end_header_id|>\n\n". If "<|eot_id|>" is not found, the extraction
    continues until the end of the string.
    """
    assistant_header = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    start_index = text.find(assistant_header)
    if start_index != -1:
        start_index += len(assistant_header)
    else:
        # Fallback: use the last occurrence of the generic header ending
        generic_header = "<|end_header_id|>\n\n"
        start_index = text.rfind(generic_header)
        if start_index != -1:
            start_index += len(generic_header)
        else:
            start_index = 0

    end_index = text.find("<|eot_id|>", start_index)
    if end_index == -1:
        end_index = len(text)
    return text[start_index:end_index].strip()

In [18]:
from unsloth.chat_templates import get_chat_template
import json

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")
FastLanguageModel.for_inference(model)

with open("/content/ML_Spring2025_HW5/test_set_evol_instruct_150.json", "r") as infile:
    test_data = json.load(infile)

inference_results = {}

for index, entry in enumerate(test_data):
    entry_id = entry.get("id", "unknown_id")
    messages = []
    for conv in entry.get("conversations", []):
        if conv.get("from") == "human":
            messages.append({"role": "user", "content": conv.get("value", "")})
        else:
            messages.append({"role": "assistant", "content": conv.get("value", "")})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    attention_mask = torch.ones_like(inputs)

    outputs = model.generate(
        input_ids=inputs,
        attention_mask=attention_mask,
        do_sample=True,
        max_new_tokens=100,
        use_cache=True,
        temperature=1.5,
        top_p=0.9,
        top_k=30,
    )

    decoded_outputs = tokenizer.batch_decode(outputs)
    parsed_outputs = [parse_true_output(output) for output in decoded_outputs]
    inference_results[entry_id] = {
        "input": messages,
        "output": parsed_outputs,
    }
    print(f"Inference completed for entry {entry_id}")

with open("pred.json", "w") as outfile:
    json.dump(inference_results, outfile, indent=4)
with open("training_config.json", "w") as outfile:
    json.dump(training_config, outfile, indent=4)

from google.colab import files
files.download("/content/pred.json")
print("Inference completed for all entries in the test set.")

Inference completed for entry identity_8174
Inference completed for entry identity_16675
Inference completed for entry identity_51749
Inference completed for entry identity_53196
Inference completed for entry identity_65799
Inference completed for entry identity_31686
Inference completed for entry identity_25291
Inference completed for entry identity_31699
Inference completed for entry identity_30359
Inference completed for entry identity_67085
Inference completed for entry identity_60450
Inference completed for entry identity_3070
Inference completed for entry identity_36778
Inference completed for entry identity_50478
Inference completed for entry identity_20143
Inference completed for entry identity_8300
Inference completed for entry identity_45513
Inference completed for entry identity_62606
Inference completed for entry identity_38166
Inference completed for entry identity_22233
Inference completed for entry identity_54369
Inference completed for entry identity_39141
Inference com

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Inference completed for all entries in the test set.


## Saving, loading finetuned models

### Save the model

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")

### Load the model

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model", # The folder path containing of the folder that contains adapter_model.safetensors, adapter_config.json and README.md
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference